# [Super AI Engineer Season 6] Hackathon Week 4
## 5 Domains Hackathon: Thai Language Image Captioning

**Super AI Engineer Season 6 - Level 1 Hackathon**  
- Dataset: Thai Language Image Captioning
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Initial Inspection  
3. Model Preparation  
4. Data Preprocessing (Prompt Setup)  
5. Evaluation & Inference  
6. Prediction & Submission Generation

# 1. Setup & Imports
### 1.1 ติดตั้งไลบรารีที่จำเป็น

In [1]:
!pip install --upgrade "pip<24.1"
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q -U accelerate bitsandbytes
!pip install -q pythainlp fairseq sacremoses datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.9 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 6.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 14.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 19.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━

### 1.2 นำเข้าไลบรารี

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import Idefics3ForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from datasets import load_dataset

# 2. Data Loading & Initial Inspection

In [3]:
import json
from datasets import load_dataset

# Path โฟลเดอร์โจทย์
base_path = '/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning'
test_path = Path(f'{base_path}/test/test')

# โหลดข้อมูลภาพรวมแบบต้นฉบับด้วย HuggingFace datasets
try:
    print("Loading datasets using HuggingFace `datasets`...")
    dataset_train = load_dataset(f"{base_path}/train/train", split="train")
    dataset_val = load_dataset(f"{base_path}/val/val", split="train")
    dataset_test = load_dataset(f"{base_path}/test/test", split="train")
    print(f"Train Dataset:\n{dataset_train}")
    print(f"Val Dataset:\n{dataset_val}")
    print(f"Test Dataset:\n{dataset_test}")
except Exception as e:
    print("Note: HuggingFace load_dataset could not find metadata. Detailed error:", e)

# กำหนด Path ตามต้นฉบับ
food_path = Path(f'{base_path}/train/train/food')
travel_path = Path(f'{base_path}/train/train/travel')

# สำรวจ Label จาก JSON ที่โจทย์ SS6 ให้มา
try:
    with open(f'{base_path}/capgen_v1.0_train.json', 'r', encoding='utf-8') as f:
        train_labels = json.load(f)
    print(f"\nLoaded {len(train_labels)} training annotations from JSON.")
except Exception as e:
    print("\nWarning: Could not load capgen_v1.0_train.json.", e)

# ตรวจสอบจำนวนไฟล์ Test สำหรับการนำไปพยากรณ์
if test_path.exists():
    # กรองเฉพาะไฟล์รูปภาพป้องกันการอ่านโฟลเดอร์ซ้อน
    image_paths = [p for p in test_path.iterdir() if p.is_file()]
    print(f"\nReady for Inference: Found {len(image_paths)} images in test_path.")
else:
    print("\nTest path not found, please check the directory structure in Kaggle.")
    image_paths = []

Loading datasets using HuggingFace `datasets`...


Resolving data files:   0%|          | 0/28004 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/4036 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/2000 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train Dataset:
Dataset({
    features: ['image', 'label'],
    num_rows: 28004
})
Val Dataset:
Dataset({
    features: ['image', 'label'],
    num_rows: 4036
})
Test Dataset:
Dataset({
    features: ['image'],
    num_rows: 2000
})

Loaded 142291 training annotations from JSON.

Ready for Inference: Found 2000 images in test_path.


# 3. Model Preparation
โหลดโมเดล `nectec/Pathumma-llm-vision-1.0.0` (Quantized 8-bit) สำหรับทำ Image Captioning ภาษาไทย

In [4]:
revision = "quantized8bit"

# Load Processor
processor = AutoProcessor.from_pretrained(
    "nectec/Pathumma-llm-vision-1.0.0",
    revision = revision,
    do_image_splitting = False,
)

# Load Model
model = Idefics3ForConditionalGeneration.from_pretrained(
    "nectec/Pathumma-llm-vision-1.0.0",
    revision = revision,     
    torch_dtype = torch.float16,
    device_map = "cuda"
)

processor_config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

Chat templates should be in a 'chat_template.jinja' file but found key='chat_template' in the processor's config. Make sure to move your template to its own file.


preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/951 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/186 [00:00<?, ?B/s]

# 4. Data Preprocessing (Prompt Setup)
เตรียม Message format แบบ Chat Template ให้รองรับกับโมเดล Idefics3 และเชื่อมรูปภาพเข้ากับ Prompt

In [5]:
question = "รายละเอียดของรูปภาพนี้"
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "You are a helpful assistant."},
            {"type": "image"},
            {"type": "text", "text": question}
        ]
    }
]

text_prompt = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
)
print("Prompt Template:\n", text_prompt)

Prompt Template:
 <|begin_of_text|>User: You are a helpful assistant.<image>รายละเอียดของรูปภาพนี้<end_of_utterance>
Assistant:


# 5. Evaluation & Inference
ฟังก์ชันรันโมเดลเพื่อพยากรณ์คำบรรยายภาพ (Caption) สำหรับข้อมูลแต่ละรูป

In [6]:
def process_image(path: Path):
    image = Image.open(path)
    
    # อ้างอิง str_id จากชื่อไฟล์ ถ้าโครงสร้างเป็น test/001.jpg ดึง 001 มาใช้งาน
    image_id = path.stem
        
    encoding = processor(
        images = image,
        text = text_prompt.strip(),
        return_tensors = "pt"
    )
    
    encoding = {k: v.to("cuda") for k, v in encoding.items()}
    model.eval()
    
    with torch.inference_mode():
        generated_ids = model.generate(
            **encoding, 
            max_new_tokens = 128,
            num_beams = 4,              # ใช้ Beam Search 4 เส้นทางช่วยค้นหาประโยคที่ดีที่สุด
            do_sample = False,          # ปิดความสุ่ม (Randomness) เพื่อให้ผลมีความสม่ำเสมอที่สุด
            repetition_penalty = 1.15,  # ป้องกันการสร้างคำซ้ำติดๆ กัน
            early_stopping = True       # ให้หยุดทันทีที่สร้างประโยคจบบริบูรณ์
        )
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens = True)[0]
    
    # ตัดเอาเฉพาะเนื้อหาภาษาไทยที่ Assistant ตอบกลับมา
    if 'Assistant:' in generated_text:
        caption_th = generated_text.split('Assistant:')[1].strip()
    else:
        caption_th = generated_text.strip()
        
    return (image_id, caption_th)

# 6. Prediction & Submission Generation
ลูปบนภาพทั้งหมด จัด Format ของ ID ตามที่โจทย์กำหนด (ล้างคราบคำว่า test/) และเซฟเป็นไฟล์ส่งผล

In [7]:
results = []

for p in tqdm(image_paths, desc="Generating Captions"):
    results.append(process_image(p))

# สร้าง DataFrame เก็บผลลัพธ์
submission = pd.DataFrame(results, columns=['image_id', 'caption'])

# จัดการกับค่าว่าง (Null/Empty String) ที่โมเดลอาจสร้างพัง แล้ว Kaggle จะฟ้อง Error ตอน Submit
# โดยเปลี่ยนเป็นค่า Defualt แทน เช่น 'รูปภาพ' หรือ 'ไม่ทราบ'
submission['caption'] = submission['caption'].replace('', 'ไม่มีคำอธิบาย')
submission['caption'] = submission['caption'].fillna('ไม่มีคำอธิบาย')

print(submission.head())

# Save เป็น CSV
submission.to_csv('5hack_tlic_submission.csv', index = False)
print("Saved 5hack_tlic_submission.csv successfully.")

Generating Captions:   0%|          | 0/2000 [00:00<?, ?it/s]

  image_id                                            caption
0    00767  แมลงที่กำลังวิ่งเล่นอยู่ตรงพื้นที่เต็มไปด้วยก้...
1    00266  อุโบสถสีขาวขนาดใหญ่ที่ตั้งอยู่ภายในวัดและมีต้น...
2    01496  เห็ดสีขาวที่ขึ้นอยู่ตรงบริเวณพื้นที่เต็มไปด้วย...
3    01600  สัตว์ที่เป็นงู 2 ตัวที่กำลังวิ่งเล่นอยู่ตรงพื้...
4    00847  ดอกไม้สีเหลืองที่อยู่บนยอดของต้นไม้ขนาดเล็กที่...
Saved 5hack_tlic_submission.csv successfully.
